**Part 1: Connect & Clean Data (Runs on local notebook memory)**

In [ ]:
# The '%' sign is crucial in Azure ML to ensure it installs directly into the active kernel
%pip install azure-ai-ml azure-identity pandas numpy

In [1]:
import pandas as pd 
import numpy as np 
from azure.ai.ml import MLClient 
from azure.identity import DefaultAzureCredential
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

In [ ]:
# connect to azure ml 
credential = DefaultAzureCredential()
ml_client = MLClient.from_config(credential = credential)

In [ ]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

# Register a fresh asset explicitly marked as a flat URI File
flat_data_config = Data(
    path="clean_churn_data.csv", # Uses the local clean file you saved in Part 1
    type=AssetTypes.URI_FILE,
    description="Flat file churn data for AutoML",
    name="churn_flat_file"
)

registered_flat_data = ml_client.data.create_or_update(flat_data_config)
print(f"Asset registered cleanly! Name: {registered_flat_data.name}, Version: {registered_flat_data.version}")

In [ ]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

# Register a fresh asset explicitly marked as a flat URI File
flat_data_config = Data(
    path="clean_churn_data.csv", # Uses the local clean file you saved in Part 1
    type=AssetTypes.URI_FILE,
    description="Flat file churn data for AutoML",
    name="churn_flat_file"
)

registered_flat_data = ml_client.data.create_or_update(flat_data_config)
print(f"Asset registered cleanly! Name: {registered_flat_data.name}, Version: {registered_flat_data.version}")

In [ ]:
# This completely creates the extensionless MLTable file in the exact path Azure expects
mltable_content = """paths:
  - file: ./clean_churn_data.csv
transformations:
  - read_delimited:
      delimiter: ','
      encoding: 'utf8'
"""

with open("MLTable", "w") as f:
    f.write(mltable_content)

print("MLTable file written successfully by Python!")

In [ ]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

# Register the local directory containing both your CSV and the new MLTable file
mltable_data_config = Data(
    path="./",  # Points to the current folder where both files sit
    type=AssetTypes.MLTABLE,
    description="Official MLTable formatted churn data",
    name="churn_mltable_asset"
)

registered_mltable = ml_client.data.create_or_update(mltable_data_config)
print(f"MLTable Asset registered successfully! Name: {registered_mltable.name}, Version: {registered_mltable.version}")

**Part 2: Budget-Constrained AutoML**

In [ ]:
from azure.ai.ml import automl
from azure.ai.ml import Input

compute_name = "zubairdost1" 

# Feed the official MLTable pointer
automl_training_data = Input(
    type="mltable", 
    path=f"azureml:{registered_mltable.name}:{registered_mltable.version}"
)

classification_job = automl.classification(
    compute=compute_name, 
    experiment_name="budget-churn-experiment",
    training_data=automl_training_data,
    target_column_name="Churn",          
    primary_metric="AUCWeighted"
)

# Enforce Azure's minimum resource requirements
classification_job.set_limits(
    timeout_minutes=15,               
    trial_timeout_minutes=5,
    max_trials=4,                     
    enable_early_termination=True
)

# Disable ensemble iterations to finish training faster
classification_job.set_training(
    enable_stack_ensemble=False,
    enable_vote_ensemble=False
)

# Submit the job to your workspace
returned_job = ml_client.jobs.create_or_update(classification_job)
print(f"Success! Job accepted by the backend framework.")
print(f"Tracking URL: {returned_job.studio_url}")

**Part 3: Test Locally & Auto-Cleanup Endpoint**

In [ ]:
from azure.ai.ml.entities import ManagedOnlineEndpoint, ManagedOnlineDeployment, Model
import numpy as np
import time

# 1. Create a completely unique endpoint name
endpoint_name = "churn-budget-ep-" + str(np.random.randint(100, 999))

print(f"Creating endpoint: {endpoint_name}...")
endpoint = ManagedOnlineEndpoint(name=endpoint_name, auth_mode="key")
ml_client.begin_create_or_update(endpoint).result()

# 2. Grab the absolute best model from your successful training job
print("Deploying the winning AutoML model...")
deployment = ManagedOnlineDeployment(
    name="budget-deploy",
    endpoint_name=endpoint_name,
    model=f"azureml://jobs/{returned_job.name}/outputs/best_model",
    instance_type="Standard_DS2_v2", # Cost-effective, lightweight testing instance
    instance_count=1
)
ml_client.online_deployments.begin_create_or_update(deployment).result()

# 3. Route 100% of the traffic to this model
endpoint.traffic = {"budget-deploy": 100}
ml_client.begin_create_or_update(endpoint).result()

print(f"\n--- SUCCESS! ---")
print(f"Your churn prediction model is live at: {endpoint.scoring_uri}")

# --- MANDATORY COST CLEANUP ---
print("\n" + "="*50)
print("CRITICAL REMINDER: Active endpoints charge by the hour.")
input("Press Enter right here to completely delete the endpoint and stop all charges...")

print("Tearing down infrastructure...")
ml_client.online_endpoints.begin_delete(name=endpoint_name).result()
print("Endpoint successfully destroyed. Everything is safe and clean!")

In [ ]:
import mlflow
import pandas as pd

# 1. Dynamically download the absolute best model artifact from your completed run
print("Downloading the winning AutoML model files locally...")
local_model_dir = ml_client.jobs.download(
    name="elated_floor_r60hsbt40z", # Your exact successful run name
    output_name="best_model", 
    download_path="./local_model"
)

# 2. Load the model directly into your notebook session using MLflow
print("Loading model into memory...")
model_uri = "./local_model/named_outputs/best_model"
loaded_model = mlflow.pyfunc.load_model(model_uri)

# 3. Create a quick dummy customer test dataframe to check predictions
# (We use the clean dataset shape, matching your specific columns)
print("Running local evaluation testing...")
test_customer = pd.DataFrame([{
    'gender': 'Female', 'SeniorCitizen': 0, 'Partner': 'Yes', 'Dependents': 'No',
    'tenure': 12, 'PhoneService': 'Yes', 'MultipleLines': 'No', 
    'InternetService': 'Fiber optic', 'OnlineSecurity': 'No', 'OnlineBackup': 'Yes',
    'DeviceProtection': 'No', 'TechSupport': 'No', 'StreamingTV': 'Yes',
    'StreamingMovies': 'No', 'Contract': 'Month-to-month', 'PaperlessBilling': 'Yes',
    'PaymentMethod': 'Electronic check', 'MonthlyCharges': 70.85, 'TotalCharges': 850.30
}])

# 4. Generate the Churn Prediction instantly
prediction = loaded_model.predict(test_customer)
print("\n" + "="*40)
print(f"🎉 LOCAL EVALUATION SUCCESSFUL!")
print(f"Predicted Customer Churn Status: {prediction[0]}")
print("="*40)